<a href="https://colab.research.google.com/github/Deepasivakumar25/SARA-Semantic-Augmented-Retrieval-Assistant/blob/main/hybrid_rag_cross_encoder_reranking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pypdf sentence-transformers faiss-cpu transformers accelerate numpy scikit-learn numpy

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from pypdf import PdfReader
reader = PdfReader("Hybrid_Search_Practice.pdf")
pdf_text = "\n".join(rec.extract_text() for rec in reader.pages)
print(pdf_text)

In [ ]:
chunk_size = 50
word_split = pdf_text.split()
chunk_list = [" ".join(word_split[i:i+chunk_size]) for i in range(0, len(word_split),chunk_size)]
print(chunk_list)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

chatbot = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
chunk_embedding = embedding_model.encode(chunk_list)
dimension = chunk_embedding.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embedding)


In [ ]:
question = "what is mean by hybrid search?"
question_embedding = embedding_model.encode([question])

distance, index_number = index.search(
    np.array(question_embedding),
    k=5
)

retrieved_chunks = [chunk_list[rec] for rec in index_number[0]]


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
x = vectorizer.fit_transform(chunk_list)

question_tfidf = vectorizer.transform([question])

similarity_scores = cosine_similarity(question_tfidf,x)

top_k = 5

top_indices = np.argsort(similarity_scores[0])[::-1][:top_k]
keyword_chunk = [chunk_list[rec] for rec in top_indices]

keyword_indices = top_indices.tolist()
semantic_indices = index_number[0].tolist()

combine = keyword_indices + semantic_indices

unique_list = list(dict.fromkeys(combine))

unique_list_text = [chunk_list[rec] for rec in unique_list]

context_list = [[question,chunk_list[rec]] for rec in unique_list]



In [ ]:
from sentence_transformers import CrossEncoder

# Load a pre-trained CrossEncoder model
cross_encoder_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Predict scores for a pair of sentences
scores = model.predict(context_list)
print(scores)
top_score_indices = np.argsort(scores)[::-1]
print(top_score_indices)

best_cross_encoders_chunks = []

for idx in top_score_indices[:3]:
    best_cross_encoders_chunks.append(unique_list_text[idx])

context = "\n\n".join(best_cross_encoders_chunks)


In [ ]:
prompt = f"""
<|user|>

Use ONLY the context below.

Context:
{context}

Question:
{question}

If the answer is not present, reply exactly:

I couldn't find that information.

<|assistant|>
"""
response = chatbot(
    prompt,
    max_new_tokens=120,
    do_sample=False,
    return_full_text=False
)

answer = response[0]["generated_text"].strip()
